# 第08课 - 多代理设计模式


## 设置


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 为什么选择多智能体系统？

现实任务如旅行规划涉及许多不同类型的专业知识——物流、本地知识、预算等。一个单一的智能体试图处理所有问题很快会变得难以驾驭。

多智能体系统通过**专业化**解决了这个问题：每个智能体专注于一个专业领域，产生比通才更高质量的结果。它们还提高了**可扩展性**——你可以添加新的智能体（例如，航班专家、餐厅评论家），而无需重写现有工作流程。智能体通过结构化的流水线组合在一起，将上下文从一个传递到下一个。


## 创建专门代理


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 构建顺序工作流

`WorkflowBuilder` 让您能够将代理连接成有向图。在这里，我们创建了一个简单的两步骤管道：**TravelPlanner** 起草行程，然后 **TravelConcierge** 审核并改进它。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 向工作流添加更多代理

多代理模式的最大优势之一就是扩展的容易性。下面我们添加一个 **BudgetReviewer** 代理，它会根据旅行者的预算检查计划，标记可能超出预算的项目，并建议节省开支的替代方案。该工作流现在依次运行三个代理：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

在本课中，您学习了如何：

1. **创建专门的代理** — 每个代理都有专注的角色（规划、礼宾、预算审核）。
2. **使用`WorkflowBuilder`和`add_edge`将代理连接到顺序工作流中**。
3. **从多代理管道中流式传输输出，跟踪哪个代理正在发言**。
4. **通过添加新代理到链中扩展工作流，而无需修改现有代理**。

多代理设计模式保持每个代理的简单性，同时产生比任何单一代理单独完成更丰富、更彻底审核的结果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：  
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译。尽管我们努力确保准确性，但请注意自动翻译可能包含错误或不准确之处。原始语言的文件应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用本翻译而产生的任何误解或误读承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
